# DeFakeX — Heterogeneous Multi-View Gated Fusion

Fresh-session Kaggle notebook implementing the corrected architecture:

- **Frequency expert:** original full image and original FFT preprocessing.
- **Spatial experts:** detected/already-cropped face only.
- **Fusion:** calibrated expert logits + projected global/local embeddings + face/context metadata.
- **No-face case:** spatial experts are masked, not given a random center crop.
- **Evaluation:** mixed held-out set, pure FF++, untouched CelebDF, and held-out phone images.

The notebook is designed to be reproducible and to reject the fused checkpoint if it does not meet balanced acceptance gates.

In [1]:
# CELL 1 — INSTALLS
!pip -q install pillow-heif facenet-pytorch pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 92.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 69.8 MB/s eta 0:00:00


In [2]:
# CELL 2 — IMPORTS AND CONFIG
import os, io, csv, gc, copy, random, hashlib
from pathlib import Path
from dataclasses import dataclass
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from pillow_heif import register_heif_opener
register_heif_opener(); ImageFile.LOAD_TRUNCATED_IMAGES=True

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import efficientnet_b3
from facenet_pytorch import MTCNN
import pyarrow.parquet as pq

from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
PIN_MEMORY=torch.cuda.is_available(); NUM_WORKERS=0
TARGET_SIZE=224; RESIZE_SIZE=256; FFT_CLIP_MAX=14.0
IMAGENET_MEAN=[0.485,0.456,0.406]; IMAGENET_STD=[0.229,0.224,0.225]
IMG_EXTS={'.jpg','.jpeg','.png','.webp','.bmp','.heic','.heif'}

FREQ_CKPT=Path('/kaggle/input/datasets/kashirhanif/frequency-model-checkpoint/best_model.pth')
ORIGINAL_SPATIAL_CKPT=Path('/kaggle/input/datasets/kashirhanif/frequency-model-checkpoint/best_model_spatial.pth')
AUXILIARY_HEAD_CKPT=Path('/kaggle/input/datasets/kashirhanif/frequency-model-checkpoint/spatial_auxiliary_multitask_head.pth')
STYLEGAN_ROOT=Path('/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train')
FLICKR_ROOT=Path('/kaggle/input/datasets/adityajn105/flickr30k/Images/flickr30k_images')
DEEPDETECT_ROOT=Path('/kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/train')
FF_SPLIT_ROOT=Path('/kaggle/input/datasets/gradientvoyager/faceforensics-c23-extracted-faces-100k/dataset_processed_split')
PHONE_ROOT=Path('/kaggle/input/datasets/kashirhanif/phone-images')
CELEB_ROOT=Path('/kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset/Celeb_V2')
OPENFAKE_DIRS=[Path(f'/kaggle/input/datasets/kashirhanif/openfake-part-{i}') for i in range(1,5)]
OPENFAKE_ROOT=Path('/kaggle/working/of_imgs'); OPENFAKE_MANIFEST=Path('/kaggle/working/of_manifest.csv')
OPENFAKE_PER_GENERATOR=1000
CAPS={'stylegan_real':25000,'stylegan_fake':25000,'flickr_real':15000,'deepdetect_fake':15000,'openfake_per_source':1000}
PHONE_TRAIN_RATIO=.60; PHONE_VAL_RATIO=.20
CELEB_REAL_CAP=10000; CELEB_FAKE_CAP=20000
FUSION_BATCH_SIZE=384; FUSION_EPOCHS=40; FUSION_LR=3e-4; FUSION_WEIGHT_DECAY=1e-4; FUSION_PATIENCE=7
PROJECTION_DIM=48; UNCERTAIN_LOW=.30; UNCERTAIN_HIGH=.70
OUTPUT_DIR=Path('/kaggle/working'); CACHE_PATH=OUTPUT_DIR/'heterogeneous_multiview_features.pt'
print('Device:',DEVICE)

Device: cuda


In [3]:
# CELL 3 — PATH CHECK
required={'frequency checkpoint':FREQ_CKPT,'spatial checkpoint':ORIGINAL_SPATIAL_CKPT,'auxiliary checkpoint':AUXILIARY_HEAD_CKPT,
          'StyleGAN':STYLEGAN_ROOT,'Flickr':FLICKR_ROOT,'DeepDetect':DEEPDETECT_ROOT,'FF++':FF_SPLIT_ROOT,
          'Phone':PHONE_ROOT,'CelebDF':CELEB_ROOT}
for name,path in required.items(): print(f'{name:<28}', 'OK' if path.exists() else 'MISSING', path)
missing=[name for name,path in required.items() if not path.exists()]
if missing: raise FileNotFoundError('Update paths in Cell 2: '+', '.join(missing))

frequency checkpoint         OK /kaggle/input/datasets/kashirhanif/frequency-model-checkpoint/best_model.pth
spatial checkpoint           OK /kaggle/input/datasets/kashirhanif/frequency-model-checkpoint/best_model_spatial.pth
auxiliary checkpoint         OK /kaggle/input/datasets/kashirhanif/frequency-model-checkpoint/spatial_auxiliary_multitask_head.pth
StyleGAN                     OK /kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train
Flickr                       OK /kaggle/input/datasets/adityajn105/flickr30k/Images/flickr30k_images
DeepDetect                   OK /kaggle/input/datasets/ayushmandatta1/deepdetect-2025/ddata/train
FF++                         OK /kaggle/input/datasets/gradientvoyager/faceforensics-c23-extracted-faces-100k/dataset_processed_split
Phone                        OK /kaggle/input/datasets/kashirhanif/phone-images
CelebDF                      OK /kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset/Celeb_V2


In [5]:
# ============================================================
# CELL 4 — REFERENCES AND SAFE HELPERS
# ============================================================

from dataclasses import dataclass
from collections import defaultdict, Counter

@dataclass(frozen=True)
class SampleRef:
    path: str
    label: int
    source: str
    scenario: str
    spatial_mode: str      # already_face | detect | unavailable
    group: str = ""


def list_images(root: Path):
    if not root.exists():
        return []

    return sorted([
        p
        for p in root.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMG_EXTS
    ])


def validate_image(path: Path):
    try:
        with Image.open(path) as image:
            image.load()
            image.convert("RGB")
        return True, ""
    except Exception as exc:
        return False, str(exc)


def cap_shuffle(paths, cap=None, seed=SEED):
    paths = list(paths)
    random.Random(seed).shuffle(paths)

    if cap is None:
        return paths

    return paths[:cap]


def group_key(path: Path):
    tokens = path.stem.replace("-", "_").split("_")

    if len(tokens) >= 2:
        return "_".join(tokens[:2])

    return path.stem


def make_refs(paths, label, source, scenario, spatial_mode):
    return [
        SampleRef(
            path=str(path),
            label=label,
            source=source,
            scenario=scenario,
            spatial_mode=spatial_mode,
            group=group_key(path),
        )
        for path in paths
    ]


def split_grouped_by_count(
    refs,
    train_ratio,
    val_ratio,
    seed=SEED,
):
    groups = defaultdict(list)

    for ref in refs:
        groups[ref.group].append(ref)

    items = list(groups.items())

    random.Random(seed).shuffle(items)

    items.sort(
        key=lambda x: len(x[1]),
        reverse=True,
    )

    total = len(refs)

    targets = {
        "train": int(total * train_ratio),
        "val": int(total * val_ratio),
        "test": (
            total
            - int(total * train_ratio)
            - int(total * val_ratio)
        ),
    }

    result = {
        "train": [],
        "val": [],
        "test": [],
    }

    for _, group_refs in items:

        deficits = {
            split: (
                targets[split]
                - len(result[split])
            )
            for split in result
        }

        chosen_split = max(
            result,
            key=lambda split:
                deficits[split]
                / max(targets[split], 1),
        )

        result[chosen_split].extend(group_refs)

    return result


def summarize(name, refs):

    print()
    print(f"{name}: {len(refs):,}")

    print(
        "labels:",
        Counter(r.label for r in refs),
    )

    print(
        "scenarios:",
        Counter(r.scenario for r in refs),
    )

    print(
        "spatial modes:",
        Counter(r.spatial_mode for r in refs),
    )

    print(
        "sources:",
        Counter(r.source for r in refs).most_common(12),
    )

In [6]:
# CELL 5 — LOAD THREE EXPERTS (PYTORCH 2.6 SAFE FOR TRUSTED FILES)
def trusted_load(path): return torch.load(path,map_location='cpu',weights_only=False)
def build_binary_efficientnet():
    m=efficientnet_b3(weights=None); n=m.classifier[1].in_features
    m.classifier=nn.Sequential(nn.Dropout(.3,inplace=True),nn.Linear(n,1)); return m

def unpack_state_dict(x):
    if not isinstance(x,dict): return x
    for k in ('model_state_dict','state_dict','model'):
        if k in x and isinstance(x[k],dict): return x[k]
    return x

def clean_state_dict(state):
    out={}
    for k,v in state.items():
        for prefix in ('module.','model.'):
            if k.startswith(prefix): k=k[len(prefix):]
        out[k]=v
    return out

def load_binary(path):
    m=build_binary_efficientnet(); ck=trusted_load(path); m.load_state_dict(clean_state_dict(unpack_state_dict(ck)),strict=True)
    m.to(DEVICE).eval(); [p.requires_grad_(False) for p in m.parameters()]; return m

class SpatialAuxiliaryHead(nn.Module):
    def __init__(self,input_dim,num_classes):
        super().__init__(); self.shared_trunk=nn.Sequential(nn.Linear(input_dim,512),nn.BatchNorm1d(512),nn.GELU(),nn.Dropout(.35),nn.Linear(512,128),nn.BatchNorm1d(128),nn.GELU(),nn.Dropout(.2)); self.binary_head=nn.Linear(128,1); self.auxiliary_head=nn.Linear(128,num_classes)
    def forward(self,x):
        z=self.shared_trunk(x); return {'binary_logit':self.binary_head(z).squeeze(1),'auxiliary_logits':self.auxiliary_head(z)}

freq_model=load_binary(FREQ_CKPT); original_spatial_model=load_binary(ORIGINAL_SPATIAL_CKPT)
freq_backbone=copy.deepcopy(freq_model); freq_backbone.classifier=nn.Identity(); freq_backbone.eval()
spatial_backbone=copy.deepcopy(original_spatial_model); spatial_backbone.classifier=nn.Identity(); spatial_backbone.eval()
aux_ck=trusted_load(AUXILIARY_HEAD_CKPT); aux_dim=int(aux_ck.get('embedding_dim',1536)); aux_names=aux_ck.get('aux_class_names',['real','fake']); aux_n=int(aux_ck.get('num_aux_classes',len(aux_names)))
aux_model=SpatialAuxiliaryHead(aux_dim,aux_n); aux_state=aux_ck.get('model_state_dict',aux_ck.get('state_dict'))
if aux_state is None: raise KeyError(f'No aux state dict. Keys={list(aux_ck)}')
aux_model.load_state_dict(clean_state_dict(aux_state),strict=True); aux_model.to(DEVICE).eval(); [p.requires_grad_(False) for p in aux_model.parameters()]
aux_mean=torch.nan_to_num(torch.as_tensor(aux_ck['feature_mean']).float(),nan=0,posinf=0,neginf=0)
aux_std=torch.nan_to_num(torch.as_tensor(aux_ck['feature_std']).float(),nan=1,posinf=1,neginf=1).clamp_min(1e-6)
print('Experts loaded.')

Experts loaded.


In [7]:
# CELL 6 — FULL-IMAGE FREQUENCY VIEW + FACE-LOCAL SPATIAL VIEW
spatial_transform=transforms.Compose([transforms.Resize((RESIZE_SIZE,RESIZE_SIZE)),transforms.CenterCrop(TARGET_SIZE),transforms.ToTensor(),transforms.Normalize(IMAGENET_MEAN,IMAGENET_STD)])

def preprocess_frequency_full(image):
    image=image.convert('RGB').resize((RESIZE_SIZE,RESIZE_SIZE),Image.Resampling.BICUBIC); o=(RESIZE_SIZE-TARGET_SIZE)//2; image=image.crop((o,o,o+TARGET_SIZE,o+TARGET_SIZE))
    arr=np.asarray(image,dtype=np.float32)/255.; x=torch.from_numpy(arr).permute(2,0,1); out=[]
    for c in x:
        f=torch.fft.fftshift(torch.fft.fft2(c)); mag=torch.log1p(torch.abs(f)); out.append(torch.clamp(mag,0,FFT_CLIP_MAX)/FFT_CLIP_MAX)
    return torch.stack(out)

mtcnn=MTCNN(keep_all=True,device=DEVICE,min_face_size=24,thresholds=[.6,.7,.7],post_process=False)

def square_crop(image,box,margin=.25):
    W,H=image.size; x1,y1,x2,y2=map(float,box); bw,bh=x2-x1,y2-y1; side=max(bw,bh)*(1+2*margin); cx,cy=(x1+x2)/2,(y1+y2)/2
    l=max(0,int(cx-side/2)); t=max(0,int(cy-side/2)); r=min(W,int(cx+side/2)); b=min(H,int(cy+side/2)); return image.crop((l,t,r,b))

def get_spatial_view(image,mode):
    if mode=='already_face': return {'crop':image,'found':1.,'confidence':1.,'count':1.,'area_ratio':1.}
    if mode=='unavailable': return {'crop':None,'found':0.,'confidence':0.,'count':0.,'area_ratio':0.}
    boxes,probs=mtcnn.detect(image)
    if boxes is None or len(boxes)==0: return {'crop':None,'found':0.,'confidence':0.,'count':0.,'area_ratio':0.}
    areas=np.array([(b[2]-b[0])*(b[3]-b[1]) for b in boxes]); i=int(areas.argmax())
    return {'crop':square_crop(image,boxes[i]),'found':1.,'confidence':float(probs[i]),'count':float(len(boxes)),'area_ratio':float(areas[i]/max(image.width*image.height,1))}

def quality_features(image):
    g=np.asarray(image.convert('L').resize((224,224)),dtype=np.float32)/255.; gx=np.diff(g,axis=1); gy=np.diff(g,axis=0)
    return np.array([g.mean(),g.std(),(gx.var()+gy.var())/2,(g<.1).mean(),(g>.9).mean(),image.width/max(image.height,1)],dtype=np.float32)

In [8]:
# CELL 7 — EXTRACT OPENFAKE FROM PARQUET IN CURRENT SESSION
GENERATORS=['sd-3.5','sd-2.1','flux.1-dev','sdxl-epic-realism','midjourney-6','gpt-image-1','sdxl','sd-1.5-dreamshaper','flux.1-schnell','dalle-3','sdxl-touchofrealism','flux-1.1-pro','sdxl-realvis-v5','ideogram-3.0','hidream-i1-full','sdxl-juggernaut','sd-1.5-epicdream','sd-1.5','flux-mvc5000','mystic','imagen-4.0','grok-2-image-1212','chroma','flux-amateursnapshotphotos','imagen-3.0-002','midjourney-7','flux-realism','recraft-v3','lumina-17-2-25','recraft-v2','aurora-20-1-25','ideogram-2.0','frames-23-1-25']

def extract_openfake(force=False):
    if OPENFAKE_MANIFEST.exists() and not force: print('Using',OPENFAKE_MANIFEST); return
    files=sorted({p for root in OPENFAKE_DIRS if root.exists() for p in root.rglob('*.parquet')})
    if not files: raise FileNotFoundError('Attach OpenFake parts 1-4.')
    OPENFAKE_ROOT.mkdir(parents=True,exist_ok=True); quotas={g:OPENFAKE_PER_GENERATOR for g in GENERATORS}; quotas['real']=OPENFAKE_PER_GENERATOR*len(GENERATORS); written=Counter(); rows=[]; idx=0
    for fp in tqdm(files,desc='Extract OpenFake'):
        if all(written[k]>=v for k,v in quotas.items()): break
        try:
            meta=pq.read_table(fp,columns=['label','model']); labels=meta['label'].to_pylist(); models=meta['model'].to_pylist()
        except Exception: continue
        wanted=[]
        for i,(lab,model) in enumerate(zip(labels,models)):
            real=str(lab).strip().lower()=='real'; src='real' if real else str(model).strip()
            if src in quotas and written[src]<quotas[src]: wanted.append((i,src,0 if real else 1))
        del meta,labels,models; gc.collect()
        if not wanted: continue
        try: table=pq.read_table(fp,columns=['image']); images=table['image']
        except Exception: continue
        for row_i,src,label in wanted:
            if written[src]>=quotas[src]: continue
            try:
                raw=images[row_i].as_py(); raw=raw.get('bytes') if isinstance(raw,dict) else raw
                if raw is None: continue
                with Image.open(io.BytesIO(raw)) as im:
                    im=im.convert('RGB'); folder=OPENFAKE_ROOT/src.replace('/','_'); folder.mkdir(parents=True,exist_ok=True); path=folder/f'{idx:08d}.jpg'; im.save(path,'JPEG',quality=88)
                rows.append({'path':str(path),'label':label,'source':src}); written[src]+=1; idx+=1
            except Exception: continue
        del images,table; gc.collect()
    pd.DataFrame(rows).to_csv(OPENFAKE_MANIFEST,index=False); print('OpenFake extracted',len(rows))
extract_openfake(False)

Extract OpenFake:   0%|          | 0/89 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 2. 
  warnings.warn(str(msg))


OpenFake extracted 62639


In [9]:
# CELL 8 — BUILD SOURCE/GROUP-AWARE SPLITS
train_refs=[]; val_refs=[]; mixed_test_refs=[]
def add_source(paths,label,source,scenario,mode,cap,seed):
    split=split_grouped_by_count(make_refs(cap_shuffle(paths,cap,seed),label,source,scenario,mode),.70,.15,seed)
    train_refs.extend(split['train']); val_refs.extend(split['val']); mixed_test_refs.extend(split['test'])

add_source(list_images(STYLEGAN_ROOT/'real'),0,'stylegan_real','clean_real','already_face',CAPS['stylegan_real'],SEED+1)
add_source(list_images(STYLEGAN_ROOT/'fake'),1,'stylegan_fake','ai_fake','already_face',CAPS['stylegan_fake'],SEED+2)
add_source(list_images(DEEPDETECT_ROOT/'fake'),1,'deepdetect_fake','ai_fake','already_face',CAPS['deepdetect_fake'],SEED+3)
add_source(list_images(FLICKR_ROOT),0,'flickr_real','clean_real','detect',CAPS['flickr_real'],SEED+4)

of=pd.read_csv(OPENFAKE_MANIFEST)
for src in sorted(of.source.unique()):
    sub=of[of.source==src]; label=int(sub.label.iloc[0]); paths=[Path(p) for p in sub.path if Path(p).exists()]; scenario='clean_real' if label==0 else 'ai_fake'; name='openfake_real' if label==0 else f'openfake_{src}'
    add_source(paths,label,name,scenario,'unavailable',CAPS['openfake_per_source'],SEED+5)

FF_TYPES=['Deepfakes','Face2Face','FaceShifter','FaceSwap','NeuralTextures','DeepFakeDetection']; pure_ff_test=[]
for split_name,target in (('train',train_refs),('val',val_refs),('test',pure_ff_test)):
    root=FF_SPLIT_ROOT/split_name
    for real_name in ('Real','real','Original','original'):
        if (root/real_name).exists(): target.extend(make_refs(list_images(root/real_name),0,'ff_Real','clean_real','already_face')); break
    for t in FF_TYPES:
        if (root/t).exists(): target.extend(make_refs(list_images(root/t),1,f'ff_{t}','deepfake_fake','already_face'))

valid_phone=[]; invalid=[]
for p in tqdm(list_images(PHONE_ROOT),desc='Validate phone'):
    ok,e=validate_image(p); valid_phone.append(p) if ok else invalid.append({'path':str(p),'error':e})
pd.DataFrame(invalid).to_csv(OUTPUT_DIR/'invalid_phone_images.csv',index=False)
phone_split=split_grouped_by_count(make_refs(valid_phone,0,'phone_real','phone_real','detect'),PHONE_TRAIN_RATIO,PHONE_VAL_RATIO,SEED)
train_refs.extend(phone_split['train']); val_refs.extend(phone_split['val']); phone_test=phone_split['test']
for refs in (train_refs,val_refs,mixed_test_refs,pure_ff_test,phone_test): random.Random(SEED).shuffle(refs)
for n,r in (('Train',train_refs),('Val',val_refs),('Mixed test',mixed_test_refs),('Pure FF++',pure_ff_test),('Phone test',phone_test)): summarize(n,r)

Validate phone:   0%|          | 0/1314 [00:00<?, ?it/s]


Train: 206,711
labels: Counter({1: 158832, 0: 47879})
scenarios: Counter({'deepfake_fake': 110086, 'ai_fake': 48746, 'clean_real': 47091, 'phone_real': 788})
spatial modes: Counter({'already_face': 173977, 'unavailable': 21446, 'detect': 11288})
sources: [('ff_DeepFakeDetection', 19880), ('ff_FaceSwap', 18885), ('ff_Real', 18391), ('ff_Face2Face', 18365), ('ff_Deepfakes', 18085), ('stylegan_real', 17500), ('stylegan_fake', 17500), ('ff_FaceShifter', 17481), ('ff_NeuralTextures', 17390), ('flickr_real', 10500), ('deepdetect_fake', 10500), ('phone_real', 788)]

Val: 44,349
labels: Counter({1: 33828, 0: 10521})
scenarios: Counter({'deepfake_fake': 23384, 'ai_fake': 10444, 'clean_real': 10259, 'phone_real': 262})
spatial modes: Counter({'already_face': 37243, 'unavailable': 4594, 'detect': 2512})
sources: [('ff_FaceSwap', 4209), ('ff_Real', 4109), ('ff_Face2Face', 4105), ('ff_Deepfakes', 4017), ('ff_FaceShifter', 3889), ('ff_NeuralTextures', 3879), ('stylegan_fake', 3750), ('stylegan_real

In [10]:
# CELL 9 — UNTOUCHED CELEBDF TEST
real=[]; fake=[]
for split in ('Train','Val','Test','train','val','test'):
    root=CELEB_ROOT/split
    if not root.exists(): continue
    for name in ('real','Real'):
        if (root/name).exists(): real.extend(list_images(root/name))
    for name in ('fake','Fake'):
        if (root/name).exists(): fake.extend(list_images(root/name))
real=cap_shuffle(real,CELEB_REAL_CAP,SEED+20); fake=cap_shuffle(fake,CELEB_FAKE_CAP,SEED+21)
celeb_test=make_refs(real,0,'celebdf_real','clean_real','already_face')+make_refs(fake,1,'celebdf_fake','deepfake_fake','already_face'); random.Random(SEED).shuffle(celeb_test); summarize('CelebDF',celeb_test)
if not real or not fake: raise RuntimeError('CelebDF folders not resolved.')


CelebDF: 30,000
labels: Counter({1: 20000, 0: 10000})
scenarios: Counter({'deepfake_fake': 20000, 'clean_real': 10000})
spatial modes: Counter({'already_face': 30000})
sources: [('celebdf_fake', 20000), ('celebdf_real', 10000)]


In [11]:
# CELL 10 — EXTRACT FULL-IMAGE / FACE-LOCAL FEATURES
@torch.inference_mode()
def extract_one(ref):
    with Image.open(ref.path) as image:
        image.load(); image=image.convert('RGB')
        freq_x=preprocess_frequency_full(image).unsqueeze(0).to(DEVICE); freq_logit=freq_model(freq_x).reshape(-1).float(); freq_emb=freq_backbone(freq_x).float()
        face=get_spatial_view(image,ref.spatial_mode)
        if face['crop'] is not None:
            sp_x=spatial_transform(face['crop']).unsqueeze(0).to(DEVICE); original_logit=original_spatial_model(sp_x).reshape(-1).float(); sp_emb=spatial_backbone(sp_x).float(); sp_cpu=torch.nan_to_num(sp_emb.cpu(),nan=0,posinf=30,neginf=-30); z=torch.nan_to_num((sp_cpu-aux_mean)/aux_std,nan=0,posinf=30,neginf=-30).clamp(-30,30); aux_logit=aux_model(z.to(DEVICE))['binary_logit'].float()
        else:
            original_logit=torch.zeros(1,device=DEVICE); aux_logit=torch.zeros(1,device=DEVICE); sp_emb=torch.zeros((1,aux_dim),device=DEVICE)
        freq_logit=torch.nan_to_num(freq_logit,nan=0,posinf=30,neginf=-30).clamp(-30,30); original_logit=torch.nan_to_num(original_logit,nan=0,posinf=30,neginf=-30).clamp(-30,30); aux_logit=torch.nan_to_num(aux_logit,nan=0,posinf=30,neginf=-30).clamp(-30,30)
        meta=np.concatenate([np.array([face['found'],face['confidence'],face['count'],face['area_ratio']],np.float32),quality_features(image)])
    return np.array([freq_logit.item(),original_logit.item(),aux_logit.item()],np.float32),torch.nan_to_num(freq_emb.cpu(),nan=0,posinf=30,neginf=-30).numpy().astype(np.float16)[0],torch.nan_to_num(sp_emb.cpu(),nan=0,posinf=30,neginf=-30).numpy().astype(np.float16)[0],meta

def extract_split(refs,name):
    L=[]; F=[]; S=[]; M=[]; kept=[]; failures=[]
    for r in tqdm(refs,desc=name):
        try:
            l,f,s,m=extract_one(r); L.append(l); F.append(f); S.append(s); M.append(m); kept.append(r)
        except Exception as e: failures.append({'path':r.path,'source':r.source,'error':str(e)})
    if failures: pd.DataFrame(failures).to_csv(OUTPUT_DIR/f'{name.lower().replace(" ","_")}_failures.csv',index=False)
    return {'expert_logits':np.asarray(L,np.float32),'freq_embedding':np.asarray(F,np.float16),'spatial_embedding':np.asarray(S,np.float16),'metadata':np.asarray(M,np.float32),'y':np.asarray([r.label for r in kept],np.int64),'sources':[r.source for r in kept],'scenarios':[r.scenario for r in kept],'paths':[r.path for r in kept]}

if CACHE_PATH.exists(): cache=torch.load(CACHE_PATH,map_location='cpu',weights_only=False); print('Loaded cache',CACHE_PATH)
else:
    cache={'train':extract_split(train_refs,'Train'),'val':extract_split(val_refs,'Val'),'mixed_test':extract_split(mixed_test_refs,'Mixed Test'),'ff_test':extract_split(pure_ff_test,'FF Test'),'celeb_test':extract_split(celeb_test,'Celeb Test'),'phone_test':extract_split(phone_test,'Phone Test')}; torch.save(cache,CACHE_PATH); print('Saved',CACHE_PATH)
for k,d in cache.items(): print(k,d['expert_logits'].shape,d['freq_embedding'].shape,d['spatial_embedding'].shape,Counter(d['scenarios']))

Train:   0%|          | 0/206711 [00:00<?, ?it/s]

Val:   0%|          | 0/44349 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=10000.0 (msgs/sec)
ServerApp.rate_limit_window=1.0 (secs)



Mixed Test:   0%|          | 0/16599 [00:00<?, ?it/s]

FF Test:   0%|          | 0/26276 [00:00<?, ?it/s]

Celeb Test:   0%|          | 0/30000 [00:00<?, ?it/s]

Phone Test:   0%|          | 0/264 [00:00<?, ?it/s]

Saved /kaggle/working/heterogeneous_multiview_features.pt
train (206711, 3) (206711, 1536) (206711, 1536) Counter({'deepfake_fake': 110086, 'ai_fake': 48746, 'clean_real': 47091, 'phone_real': 788})
val (44349, 3) (44349, 1536) (44349, 1536) Counter({'deepfake_fake': 23384, 'ai_fake': 10444, 'clean_real': 10259, 'phone_real': 262})
mixed_test (16599, 3) (16599, 1536) (16599, 1536) Counter({'ai_fake': 10449, 'clean_real': 6150})
ff_test (26276, 3) (26276, 1536) (26276, 1536) Counter({'deepfake_fake': 22332, 'clean_real': 3944})
celeb_test (30000, 3) (30000, 1536) (30000, 1536) Counter({'deepfake_fake': 20000, 'clean_real': 10000})
phone_test (264, 3) (264, 1536) (264, 1536) Counter({'phone_real': 264})


In [12]:
# CELL 11 — LEAKAGE / FINITE AUDIT
def H(paths): return {hashlib.sha1(str(Path(p).resolve()).encode()).hexdigest() for p in paths}
hashes={k:H(v['paths']) for k,v in cache.items()}
for a,b in (('train','val'),('train','mixed_test'),('train','ff_test'),('train','celeb_test'),('train','phone_test'),('val','phone_test')):
    overlap=hashes[a]&hashes[b]; print(a,b,len(overlap));
    if overlap: raise RuntimeError(f'Leakage {a} {b}')
for split,d in cache.items():
    mask=np.ones(len(d['y']),dtype=bool)
    for key in ('expert_logits','freq_embedding','spatial_embedding','metadata'): mask &= np.isfinite(np.asarray(d[key])).all(1)
    print(split,'invalid rows',int((~mask).sum()))
    if not mask.all():
        for key in ('expert_logits','freq_embedding','spatial_embedding','metadata','y'): d[key]=np.asarray(d[key])[mask]
        for key in ('sources','scenarios','paths'): d[key]=[x for x,k in zip(d[key],mask) if k]

train val 0
train mixed_test 0
train ff_test 0
train celeb_test 0
train phone_test 0
val phone_test 0
train invalid rows 0
val invalid rows 0
mixed_test invalid rows 0
ff_test invalid rows 0
celeb_test invalid rows 0
phone_test invalid rows 0


In [13]:
# CELL 12 — TEMPERATURE CALIBRATION OF EACH EXPERT
class Temp(nn.Module):
    def __init__(self): super().__init__(); self.log_t=nn.Parameter(torch.zeros(1))
    def forward(self,x): return x/self.log_t.exp().clamp(.05,20)
def fit_temp(logits,y):
    x=torch.as_tensor(logits).float().to(DEVICE); y=torch.as_tensor(y).float().to(DEVICE); m=Temp().to(DEVICE); opt=torch.optim.LBFGS(m.parameters(),lr=.1,max_iter=100); lossfn=nn.BCEWithLogitsLoss()
    def closure(): opt.zero_grad(); loss=lossfn(m(x),y); loss.backward(); return loss
    opt.step(closure); return float(m.log_t.exp().detach().cpu())
temperatures=[fit_temp(cache['val']['expert_logits'][:,i],cache['val']['y']) for i in range(3)]; print('Temperatures',temperatures)
def calibrated_logits(split):
    x=cache[split]['expert_logits'].astype(np.float32).copy()
    for i,t in enumerate(temperatures): x[:,i]/=t
    return x

Temperatures [5.746281147003174, 5.97663688659668, 3.817824125289917]


In [14]:
# CELL 13 — NORMALIZATION AND TENSOR PREPARATION
fm=cache['train']['freq_embedding'].astype(np.float32).mean(0,keepdims=True); fs=cache['train']['freq_embedding'].astype(np.float32).std(0,keepdims=True).clip(1e-6)
sm=cache['train']['spatial_embedding'].astype(np.float32).mean(0,keepdims=True); ss=cache['train']['spatial_embedding'].astype(np.float32).std(0,keepdims=True).clip(1e-6)
meta_scaler=StandardScaler().fit(cache['train']['metadata'])
def prep(split):
    d=cache[split]; return {'logits':calibrated_logits(split),'freq':((d['freq_embedding'].astype(np.float32)-fm)/fs).astype(np.float32),'spatial':((d['spatial_embedding'].astype(np.float32)-sm)/ss).astype(np.float32),'meta':meta_scaler.transform(d['metadata']).astype(np.float32),'raw_meta':d['metadata'].astype(np.float32),'y':d['y'].astype(np.float32),'scenarios':d['scenarios']}
train_data=prep('train'); val_data=prep('val')

In [15]:
# CELL 14 — GATED MULTI-VIEW MIXTURE OF EXPERTS
class FD(Dataset):
    def __init__(self,d): self.l=torch.from_numpy(d['logits']); self.f=torch.from_numpy(d['freq']); self.s=torch.from_numpy(d['spatial']); self.m=torch.from_numpy(d['meta']); self.r=torch.from_numpy(d['raw_meta']); self.y=torch.from_numpy(d['y']); self.sc=d['scenarios']
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.l[i],self.f[i],self.s[i],self.m[i],self.r[i],self.y[i]
class Gate(nn.Module):
    def __init__(self,fd,sd,md,p=48):
        super().__init__(); self.fp=nn.Sequential(nn.Linear(fd,128),nn.GELU(),nn.Dropout(.15),nn.Linear(128,p),nn.GELU()); self.sp=nn.Sequential(nn.Linear(sd,128),nn.GELU(),nn.Dropout(.15),nn.Linear(128,p),nn.GELU()); c=3+2*p+md; self.g=nn.Sequential(nn.Linear(c,96),nn.GELU(),nn.Dropout(.2),nn.Linear(96,3)); self.r=nn.Sequential(nn.Linear(c,64),nn.GELU(),nn.Dropout(.15),nn.Linear(64,1))
    def forward(self,l,f,s,m,raw):
        c=torch.cat([l,self.fp(f),self.sp(s),m],1); w=torch.softmax(self.g(c),1); available=raw[:,0:1].clamp(0,1); mask=torch.cat([torch.ones_like(available),available,available],1); w=w*mask; w=w/w.sum(1,keepdim=True).clamp_min(1e-6); return (w*l).sum(1)+.25*self.r(c).squeeze(1),w
counts=Counter(train_data['scenarios']); sw=np.array([1/counts[x] for x in train_data['scenarios']],np.float64); sampler=WeightedRandomSampler(sw,len(sw),replacement=True)
train_loader=DataLoader(FD(train_data),batch_size=FUSION_BATCH_SIZE,sampler=sampler); val_loader=DataLoader(FD(val_data),batch_size=FUSION_BATCH_SIZE,shuffle=False)
model=Gate(train_data['freq'].shape[1],train_data['spatial'].shape[1],train_data['meta'].shape[1],PROJECTION_DIM).to(DEVICE); print(model)

Gate(
  (fp): Sequential(
    (0): Linear(in_features=1536, out_features=128, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.15, inplace=False)
    (3): Linear(in_features=128, out_features=48, bias=True)
    (4): GELU(approximate='none')
  )
  (sp): Sequential(
    (0): Linear(in_features=1536, out_features=128, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.15, inplace=False)
    (3): Linear(in_features=128, out_features=48, bias=True)
    (4): GELU(approximate='none')
  )
  (g): Sequential(
    (0): Linear(in_features=109, out_features=96, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=96, out_features=3, bias=True)
  )
  (r): Sequential(
    (0): Linear(in_features=109, out_features=64, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.15, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [16]:
# CELL 15 — METRICS / THRESHOLD / INFERENCE
def metrics(y,p,t):
    z=(p>=t).astype(int); return {'accuracy':accuracy_score(y,z),'macro_f1':f1_score(y,z,average='macro',zero_division=0),'real_f1':f1_score(y,z,pos_label=0,zero_division=0),'fake_f1':f1_score(y,z,pos_label=1,zero_division=0),'real_recall':recall_score(y,z,pos_label=0,zero_division=0),'fake_recall':recall_score(y,z,pos_label=1,zero_division=0),'auc':roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan,'pred':z}
def scenario_recalls(y,p,t,sc):
    z=(p>=t).astype(int); sc=np.asarray(sc); out={}
    for s in sorted(set(sc)):
        m=sc==s; lab=int(round(y[m].mean())); out[s]=recall_score(y[m],z[m],pos_label=lab,zero_division=0)
    return out
def choose_threshold(y,p,sc):
    rows=[]
    for t in np.linspace(.05,.95,361):
        m=metrics(y,p,t); sr=scenario_recalls(y,p,t,sc); worst=min(sr.values()); gap=abs(m['real_recall']-m['fake_recall']); score=.4*m['macro_f1']+.35*worst+.2*min(m['real_recall'],m['fake_recall'])-.15*gap; rows.append({'threshold':t,'score':score,**{k:v for k,v in m.items() if k!='pred'},'worst_scenario':worst,**{f'recall_{k}':v for k,v in sr.items()}})
    df=pd.DataFrame(rows); return df.loc[df.score.idxmax()],df
@torch.inference_mode()
def infer(d):
    loader=DataLoader(FD(d),batch_size=1024,shuffle=False); P=[]; W=[]; model.eval()
    for l,f,s,m,r,_ in loader:
        q,w=model(l.to(DEVICE),f.to(DEVICE),s.to(DEVICE),m.to(DEVICE),r.to(DEVICE)); P.extend(torch.sigmoid(q).cpu().numpy()); W.append(w.cpu().numpy())
    return np.asarray(P),np.concatenate(W)

In [17]:
# CELL 16 — TRAIN
lossfn=nn.BCEWithLogitsLoss(); opt=torch.optim.AdamW(model.parameters(),lr=FUSION_LR,weight_decay=FUSION_WEIGHT_DECAY); scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='max',factor=.5,patience=2,min_lr=1e-6)
best=None; best_score=-np.inf; stale=0; best_epoch=0; history=[]
for epoch in range(1,FUSION_EPOCHS+1):
    model.train(); total=0; n=0
    for l,f,s,m,r,y in train_loader:
        l,f,s,m,r,y=[x.to(DEVICE) for x in (l,f,s,m,r,y)]; opt.zero_grad(set_to_none=True); q,w=model(l,f,s,m,r); bce=lossfn(q,y); entropy=-(w.clamp_min(1e-8)*w.clamp_min(1e-8).log()).sum(1).mean(); loss=bce-.01*entropy; loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5); opt.step(); total+=loss.item()*len(y); n+=len(y)
    vp,vw=infer(val_data); row,_=choose_threshold(val_data['y'],vp,val_data['scenarios']); score=float(row.score); scheduler.step(score); print(f'Epoch {epoch:02d} loss={total/max(n,1):.4f} score={score:.4f} macro={row.macro_f1:.4f} real={row.real_recall:.4f} fake={row.fake_recall:.4f} worst={row.worst_scenario:.4f} gates={vw.mean(0).round(3)}'); history.append({'epoch':epoch,'loss':total/max(n,1),**row.to_dict()})
    if score>best_score: best_score=score; best_epoch=epoch; best=copy.deepcopy(model.state_dict()); stale=0
    else:
        stale+=1
        if stale>=FUSION_PATIENCE: print('Early stopping'); break
model.load_state_dict(best); vp,vw=infer(val_data); threshold_row,threshold_table=choose_threshold(val_data['y'],vp,val_data['scenarios']); FUSION_THRESHOLD=float(threshold_row.threshold); print('Best epoch',best_epoch); display(threshold_row.to_frame().T)

Epoch 01 loss=0.1403 score=0.8617 macro=0.9188 real=0.9101 fake=0.9490 worst=0.9086 gates=[0.422 0.28  0.298]
Epoch 02 loss=0.0710 score=0.8740 macro=0.9260 real=0.9239 fake=0.9515 worst=0.9227 gates=[0.425 0.266 0.309]
Epoch 03 loss=0.0561 score=0.8829 macro=0.9306 real=0.9341 fake=0.9527 worst=0.9331 gates=[0.414 0.286 0.301]
Epoch 04 loss=0.0476 score=0.8807 macro=0.9307 real=0.9311 fake=0.9539 worst=0.9304 gates=[0.425 0.279 0.297]
Epoch 05 loss=0.0436 score=0.8811 macro=0.9300 real=0.9319 fake=0.9529 worst=0.9312 gates=[0.416 0.287 0.297]
Epoch 06 loss=0.0385 score=0.8760 macro=0.9286 real=0.9256 fake=0.9536 worst=0.9247 gates=[0.413 0.284 0.302]
Epoch 07 loss=0.0335 score=0.8837 macro=0.9299 real=0.9384 fake=0.9504 worst=0.9310 gates=[0.417 0.286 0.298]
Epoch 08 loss=0.0305 score=0.8794 macro=0.9288 real=0.9349 fake=0.9505 worst=0.9237 gates=[0.415 0.28  0.305]
Epoch 09 loss=0.0299 score=0.8808 macro=0.9310 real=0.9308 fake=0.9543 worst=0.9308 gates=[0.416 0.286 0.298]
Epoch 10 l

,threshold,score,accuracy,macro_f1,real_f1,fake_f1,real_recall,fake_recall,auc,worst_scenario,recall_ai_fake,recall_clean_real,recall_deepfake_fake,recall_phone_real
325,0.8625,0.883663,0.947552,0.929853,0.894618,0.965089,0.938409,0.950396,0.981649,0.930965,0.930965,0.938493,0.959075,0.935115


In [19]:
# CELL 17 — FINAL EVALUATION / GATE ANALYSIS
def report(title,d,p,t,w):
    m=metrics(d['y'],p,t); print(''+'='*72+''+title+''+'='*72); print('threshold',t); 
    [print(f'{k:<13}: {m[k]:.4f}') for k in ('accuracy','macro_f1','real_f1','fake_f1','real_recall','fake_recall','auc')]; print(confusion_matrix(d['y'],m['pred'],labels=[0,1])); print('scenario recalls',scenario_recalls(d['y'],p,t,d['scenarios'])); print('mean gates',w.mean(0).round(4)); return m
rows=[]; gate_rows=[]
for split,title in (('mixed_test','MIXED HELD-OUT'),('ff_test','PURE FF++'),('celeb_test','CELEBDF ZERO-SHOT'),('phone_test','PHONE HELD-OUT')):
    d=prep(split); p,w=infer(d); m=report(title,d,p,FUSION_THRESHOLD,w); rows.append({'split':split,**{k:v for k,v in m.items() if k!='pred'}}); sc=np.asarray(d['scenarios'])
    for s in sorted(set(sc)):
        z=sc==s; gate_rows.append({'split':split,'scenario':s,'n':int(z.sum()),'frequency':w[z,0].mean(),'original_spatial':w[z,1].mean(),'auxiliary_spatial':w[z,2].mean()})
results=pd.DataFrame(rows); gates=pd.DataFrame(gate_rows); display(results); display(gates); results.to_csv(OUTPUT_DIR/'heterogeneous_fusion_results.csv',index=False); gates.to_csv(OUTPUT_DIR/'heterogeneous_fusion_gates.csv',index=False)

========================================================================MIXED HELD-OUT========================================================================
threshold 0.8624999999999999
accuracy     : 0.9411
macro_f1     : 0.9381
real_f1      : 0.9245
fake_f1      : 0.9517
real_recall  : 0.9741
fake_recall  : 0.9216
auc          : 0.9907
[[5991  159]
 [ 819 9630]]
scenario recalls {np.str_('ai_fake'): 0.921619293712317, np.str_('clean_real'): 0.9741463414634146}
mean gates [0.5724 0.2162 0.2114]
========================================================================PURE FF++========================================================================
threshold 0.8624999999999999
accuracy     : 0.9524
macro_f1     : 0.9091
real_f1      : 0.8462
fake_f1      : 0.9719
real_recall  : 0.8722
fake_recall  : 0.9666
auc          : 0.9809
[[ 3440   504]
 [  746 21586]]
scenario recalls {np.str_('clean_real'): 0.8722109533468559, np.str_('deepfake_fake'): 0.9665950205982446}
mean gates [0.3241 0.3

,split,accuracy,macro_f1,real_f1,fake_f1,real_recall,fake_recall,auc
0,mixed_test,0.941081,0.938106,0.924537,0.951675,0.974146,0.921619,0.990730
1,ff_test,0.952428,0.909055,0.846248,0.971861,0.872211,0.966595,0.980858
2,celeb_test,0.711133,0.674976,0.566570,0.783382,0.566400,0.783500,0.739840
3,phone_test,0.946970,0.486381,0.972763,0.000000,0.946970,0.000000,NaN


,split,scenario,n,frequency,original_spatial,auxiliary_spatial
0,mixed_test,ai_fake,10449,0.638435,0.185514,0.176051
1,mixed_test,clean_real,6150,0.460125,0.268388,0.271488
2,ff_test,clean_real,3944,0.312441,0.317017,0.370542
3,ff_test,deepfake_fake,22332,0.326155,0.330353,0.343492
4,celeb_test,clean_real,10000,0.281781,0.315595,0.402624
5,celeb_test,deepfake_fake,20000,0.292202,0.323501,0.384297
6,phone_test,phone_real,264,0.429913,0.268286,0.301801


In [20]:
# CELL 18 — UNCERTAINTY ANALYSIS
u=[]
for split in ('mixed_test','ff_test','celeb_test','phone_test'):
    d=prep(split); p,_=infer(d); certain=(p<=UNCERTAIN_LOW)|(p>=UNCERTAIN_HIGH); acc=accuracy_score(d['y'][certain],(p[certain]>=.5).astype(int)) if certain.any() else np.nan; u.append({'split':split,'coverage':certain.mean(),'certain_accuracy':acc,'uncertain_rate':1-certain.mean()})
uncertainty=pd.DataFrame(u); display(uncertainty); uncertainty.to_csv(OUTPUT_DIR/'fusion_uncertainty.csv',index=False)

,split,coverage,certain_accuracy,uncertain_rate
0,mixed_test,0.964938,0.973154,0.035062
1,ff_test,0.988469,0.959727,0.011531
2,celeb_test,0.932300,0.745504,0.067700
3,phone_test,0.958333,0.944664,0.041667


In [21]:
# CELL 19 — SAVE PACKAGE / ACCEPTANCE GATES
package={'fusion_state_dict':model.state_dict(),'threshold':FUSION_THRESHOLD,'temperatures':temperatures,'freq_mean':fm,'freq_std':fs,'spatial_mean':sm,'spatial_std':ss,'metadata_mean':meta_scaler.mean_,'metadata_scale':meta_scaler.scale_,'projection_dim':PROJECTION_DIM,'best_epoch':best_epoch,'best_validation_score':best_score,'uncertain_low':UNCERTAIN_LOW,'uncertain_high':UNCERTAIN_HIGH}
torch.save(package,OUTPUT_DIR/'defakex_heterogeneous_gated_fusion.pth'); pd.DataFrame(history).to_csv(OUTPUT_DIR/'fusion_training_history.csv',index=False); threshold_table.to_csv(OUTPUT_DIR/'fusion_threshold_sweep.csv',index=False)
r=results.set_index('split'); checks={'FF++ macro-F1 >= .90':r.loc['ff_test','macro_f1']>=.90,'FF++ real recall >= .88':r.loc['ff_test','real_recall']>=.88,'FF++ fake recall >= .92':r.loc['ff_test','fake_recall']>=.92,'CelebDF AUC >= .76':r.loc['celeb_test','auc']>=.76,'CelebDF real recall >= .70':r.loc['celeb_test','real_recall']>=.70,'CelebDF fake recall >= .70':r.loc['celeb_test','fake_recall']>=.70,'Phone real recall >= .80':r.loc['phone_test','real_recall']>=.80}
for k,v in checks.items(): print('PASS' if v else 'FAIL','-',k)
print('Passed',sum(checks.values()),'/',len(checks)); print('Saved',OUTPUT_DIR/'defakex_heterogeneous_gated_fusion.pth')

PASS - FF++ macro-F1 >= .90
FAIL - FF++ real recall >= .88
PASS - FF++ fake recall >= .92
FAIL - CelebDF AUC >= .76
FAIL - CelebDF real recall >= .70
PASS - CelebDF fake recall >= .70
PASS - Phone real recall >= .80
Passed 4 / 7
Saved /kaggle/working/defakex_heterogeneous_gated_fusion.pth
